Procesamiento de datos con Pandas


A1 Diagnsostico


In [3]:
from pathlib import Path
import pandas as pd

RUTA = Path("ventas.csv")   # relativa: busca el archivo junto al notebook
ventas = pd.read_csv(RUTA)

In [4]:
print(ventas.shape)
print(ventas.dtypes)
print(ventas.isna().sum())
ventas.head()

(150, 6)
fecha               object
region              object
producto            object
unidades            object
precio_unitario    float64
cliente_id          object
dtype: object
fecha              0
region             0
producto           0
unidades           2
precio_unitario    1
cliente_id         0
dtype: int64


,fecha,region,producto,unidades,precio_unitario,cliente_id
0,2026-01-09,Sur,Cable Kit,7,12.50,C005
1,2026-03-14,Oeste,Gateway,25,120.00,C032
2,2026-01-26,Oeste,Hub IoT,1,88.96,C016
3,2026-02-05,Norte,Gateway,0,120.00,C010
4,2026-02-07,Norte,Hub IoT,15,85.00,C027


In [5]:
ventas.describe()

,precio_unitario
count,149.000000
mean,51.281678
std,42.478075
min,-32.000000
25%,19.500000
50%,32.000000
75%,85.000000
max,120.000000


In [7]:
unidades_num = pd.to_numeric(ventas["unidades"], errors="coerce")
precio_num = pd.to_numeric(ventas["precio_unitario"], errors="coerce")

In [8]:
sospechosas = ventas[
    unidades_num.isna() | precio_num.isna() | (unidades_num <= 0) | (precio_num <= 0)
]
print(len(sospechosas))
sospechosas

10


,fecha,region,producto,unidades,precio_unitario,cliente_id
3,2026-02-05,Norte,Gateway,0,120.0,C010
41,2026-02-09,Oeste,Cable Kit,4,0.0,C015
54,2026-02-01,Sur,Sensor B,NaN,32.0,C004
63,2026-02-11,Norte,Sensor A,NaN,19.5,C018
66,2026-02-17,Oeste,Hub IoT,5,NaN,C025
84,2026-02-15,Este,Gateway,na,120.0,C022
88,2026-02-07,Este,Hub IoT,-2,85.0,C012
109,2026-02-19,Norte,Cable Kit,0,-1.0,C030
134,2026-02-03,Sur,Sensor A,3,-5.0,C007
149,2026-02-13,Sur,Sensor B,8,-32.0,C020


In [10]:
from typing import Tuple

def validar_ventas(frame: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    frame = frame.copy()

    # Regla 1: unidades numérica y > 0
    frame["unidades"] = pd.to_numeric(frame["unidades"], errors="coerce")
    # Regla 2: precio_unitario numérico y > 0
    frame["precio_unitario"] = pd.to_numeric(frame["precio_unitario"], errors="coerce")

    es_valida = (frame["unidades"] > 0) & (frame["precio_unitario"] > 0)
    validos = frame[es_valida].copy()
    errores = frame[~es_valida]

    # Regla 3: importe solo en las válidas
    validos["importe"] = validos["unidades"] * validos["precio_unitario"]
    return validos, errores

In [11]:
validos, errores = validar_ventas(ventas)
print(len(validos), len(errores))

140 10


In [13]:
importe_por_region = validos.groupby("region")["importe"].sum().sort_values(ascending=False)
print(importe_por_region)

region
Este     33122.53
Norte    22914.63
Sur      21396.35
Oeste    19164.66
Name: importe, dtype: float64


In [14]:
top3_productos = validos.groupby("producto")["importe"].sum().sort_values(ascending=False).head(3)
print(top3_productos)

producto
Gateway     40440.00
Hub IoT     33058.66
Sensor B    11568.66
Name: importe, dtype: float64


In [15]:
compras_por_cliente = validos["cliente_id"].value_counts()
clientes_repetidos = compras_por_cliente[compras_por_cliente > 1]
print(len(clientes_repetidos))
print(clientes_repetidos)

34
C027    9
C004    8
C032    8
C035    7
C013    6
C009    6
C005    5
C022    5
C038    5
C010    4
C024    4
C018    4
C034    4
C040    4
C023    4
C025    4
C037    4
C015    4
C002    4
C030    3
C007    3
C033    3
C011    3
C029    3
C003    3
C016    3
C031    2
C028    2
C001    2
C017    2
C036    2
C021    2
C020    2
C039    2
Name: cliente_id, dtype: int64


In [16]:
import json
from pathlib import Path

validos.to_csv(Path("ventas_limpias.csv"), index=False)

calidad = {
    "filas_totales": len(ventas),
    "filas_validas": len(validos),
    "filas_invalidas": len(errores),
    "importe_total": round(float(validos["importe"].sum()), 2),
}

with open(Path("calidad_ventas.json"), "w", encoding="utf-8") as f:
    json.dump(calidad, f, indent=2)

print(Path("calidad_ventas.json").read_text(encoding="utf-8"))

{
  "filas_totales": 150,
  "filas_validas": 140,
  "filas_invalidas": 10,
  "importe_total": 96598.17
}


Penguins

In [20]:
from pathlib import Path
import pandas as pd

RUTA = Path("penguins.csv")   # relativa: busca el archivo junto al notebook
penguins = pd.read_csv(RUTA)

In [22]:
print(penguins.shape)
print(penguins.dtypes)
print(penguins.isna().sum())
penguins.head()

(344, 7)
species               object
island                object
bill_length_mm       float64
bill_depth_mm        float64
flipper_length_mm    float64
body_mass_g          float64
sex                   object
dtype: object
species               0
island                0
bill_length_mm        2
bill_depth_mm         2
flipper_length_mm     2
body_mass_g           2
sex                  11
dtype: int64


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,MALE
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,FEMALE
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,FEMALE
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,FEMALE


In [23]:
cols_num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]

medidas_ok = (penguins[cols_num] > 0).all(axis=1)      # las 4 medidas > 0 (NaN > 0 es False)
sexo_ok = penguins["sex"].isin(["MALE", "FEMALE"])
especie_isla_ok = penguins["species"].notna() & penguins["island"].notna()

potenciales = penguins[~(medidas_ok & sexo_ok & especie_isla_ok)]
print(len(potenciales))
potenciales

11


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
3,Adelie,Torgersen,NaN,NaN,NaN,NaN,NaN
8,Adelie,Torgersen,34.1,18.1,193.0,3475.0,NaN
9,Adelie,Torgersen,42.0,20.2,190.0,4250.0,NaN
10,Adelie,Torgersen,37.8,17.1,186.0,3300.0,NaN
11,Adelie,Torgersen,37.8,17.3,180.0,3700.0,NaN
47,Adelie,Dream,37.5,18.9,179.0,2975.0,NaN
246,Gentoo,Biscoe,44.5,14.3,216.0,4100.0,NaN
286,Gentoo,Biscoe,46.2,14.4,214.0,4650.0,NaN
324,Gentoo,Biscoe,47.3,13.8,216.0,4725.0,NaN
336,Gentoo,Biscoe,44.5,15.7,217.0,4875.0,NaN


In [24]:
from typing import Tuple

def validar_penguins(frame: pd.DataFrame) -> Tuple[pd.DataFrame, pd.DataFrame]:
    frame = frame.copy()

    # Regla 1: las 4 medidas numéricas y > 0
    cols_num = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
    for col in cols_num:
        frame[col] = pd.to_numeric(frame[col], errors="coerce")
    medidas_ok = (frame[cols_num] > 0).all(axis=1)

    # Regla 2: sex en {MALE, FEMALE}, normalizando a mayúsculas
    frame["sex"] = frame["sex"].str.upper()
    sexo_ok = frame["sex"].isin(["MALE", "FEMALE"])

    # Regla 3: species e island no nulos
    especie_isla_ok = frame["species"].notna() & frame["island"].notna()

    es_valida = medidas_ok & sexo_ok & especie_isla_ok
    return frame[es_valida].copy(), frame[~es_valida].copy()


validos_p, errores_p = validar_penguins(penguins)
print(len(validos_p), len(errores_p))

333 11


Agregaciones

In [25]:
masa_media_especie = validos_p.groupby("species")["body_mass_g"].mean().round(1)
print(masa_media_especie)

conteo_isla_sexo = validos_p.groupby(["island", "sex"]).size()
print(conteo_isla_sexo)

top5_aleta = validos_p.nlargest(5, "flipper_length_mm")
print(top5_aleta)

species
Adelie       3706.2
Chinstrap    3733.1
Gentoo       5092.4
Name: body_mass_g, dtype: float64
island     sex   
Biscoe     FEMALE    80
           MALE      83
Dream      FEMALE    61
           MALE      62
Torgersen  FEMALE    24
           MALE      23
dtype: int64
    species  island  bill_length_mm  bill_depth_mm  flipper_length_mm  \
283  Gentoo  Biscoe            54.3           15.7              231.0   
221  Gentoo  Biscoe            50.0           16.3              230.0   
253  Gentoo  Biscoe            59.6           17.0              230.0   
285  Gentoo  Biscoe            49.8           16.8              230.0   
295  Gentoo  Biscoe            48.6           16.0              230.0   

     body_mass_g   sex  
283       5650.0  MALE  
221       5700.0  MALE  
253       6050.0  MALE  
285       5700.0  MALE  
295       5800.0  MALE  


In [26]:
import json
from pathlib import Path

validos_p.to_csv(Path("penguins_limpios.csv"), index=False)

calidad_p = {
    "filas_totales": len(penguins),
    "filas_validas": len(validos_p),
    "filas_invalidas": len(errores_p),
    "body_mass_mean": round(float(validos_p["body_mass_g"].mean()), 2),
}

with open(Path("calidad_penguins.json"), "w", encoding="utf-8") as f:
    json.dump(calidad_p, f, indent=2)

print(Path("calidad_penguins.json").read_text(encoding="utf-8"))

{
  "filas_totales": 344,
  "filas_validas": 333,
  "filas_invalidas": 11,
  "body_mass_mean": 4207.06
}
